<a href="https://www.kaggle.com/code/muhammaddhiyaulatha/arc-baseline-zero-model-ipynb?scriptVersionId=312379977" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# ARC Prize 2026 - Baseline Model

This notebook contains my first submission to the ARC-AGI-2 competition on Kaggle.

## Approach

* Generate output grids filled with zeros
* Match input grid dimensions
* Ensure correct submission format

## Purpose

This is a baseline to understand:

* Submission pipeline
* Evaluation system
* Dataset structure

Next step: implement rule-based reasoning.


In [1]:
import json
import numpy as np
from collections import Counter
from scipy.ndimage import label

class ARCSolver:
    def __init__(self):
        self.rules = [
            self.identity, self.rotate_90, self.rotate_180, self.rotate_270,
            self.flip_h, self.flip_v, self.transpose,
            self.tile_2x2, self.scale_2x,
            self.crop_to_content, self.fill_background,
            self.denoise, self.complete_symmetry,
            self.transform_largest_object
        ]

    # ======================
    # OBJECT DETECTION
    # ======================
    def extract_objects(self, x):
        objects = []
        for color in np.unique(x):
            if color == 0:
                continue

            mask = (x == color)
            structure = np.array([[0,1,0],
                                  [1,1,1],
                                  [0,1,0]])
            labeled, n = label(mask, structure=structure)

            for i in range(1, n+1):
                obj = (labeled == i)
                coords = np.argwhere(obj)
                x1,y1 = coords.min(0)
                x2,y2 = coords.max(0)

                cropped = x[x1:x2+1, y1:y2+1]
                objects.append((color, cropped, (x1,y1,x2,y2)))

        return objects

    def transform_largest_object(self, x):
        objs = self.extract_objects(x)
        if not objs:
            return x

        largest = max(objs, key=lambda o: o[1].size)
        color, obj, (x1,y1,x2,y2) = largest

        new_obj = np.rot90(obj)

        res = x.copy()
        res[x1:x2+1, y1:y2+1] = 0

        h, w = new_obj.shape
        res[x1:x1+h, y1:y1+w] = new_obj

        return res

    # ======================
    # SYMMETRY
    # ======================
    def complete_symmetry(self, x):
        h = np.hstack([x, np.flip(x, 1)])
        v = np.vstack([x, np.flip(x, 0)])
        return h if abs(h.shape[0]-h.shape[1]) < abs(v.shape[0]-v.shape[1]) else v

    # ======================
    # COLOR MAP
    # ======================
    def get_color_map(self, inp, out):
        inp, out = np.array(inp), np.array(out)
        if inp.shape != out.shape:
            return None

        mapping = {}
        for i in range(inp.shape[0]):
            for j in range(inp.shape[1]):
                a, b = inp[i,j], out[i,j]
                if a in mapping and mapping[a] != b:
                    return None
                mapping[a] = b
        return mapping

    def apply_color_map(self, x, m):
        res = x.copy()
        for k, v in m.items():
            res[x == k] = v
        return res

    # ======================
    # BASIC OPS
    # ======================
    def identity(self, x): return x
    def rotate_90(self, x): return np.rot90(x, -1)
    def rotate_180(self, x): return np.rot90(x, -2)
    def rotate_270(self, x): return np.rot90(x, -3)
    def flip_h(self, x): return np.flip(x, 1)
    def flip_v(self, x): return np.flip(x, 0)
    def transpose(self, x): return x.T
    def tile_2x2(self, x): return np.tile(x, (2,2))
    def scale_2x(self, x): return np.repeat(np.repeat(x,2,0),2,1)

    def crop_to_content(self, x):
        coords = np.argwhere(x != 0)
        if coords.size == 0: return x
        x1,y1 = coords.min(0)
        x2,y2 = coords.max(0)
        return x[x1:x2+1, y1:y2+1]

    def fill_background(self, x):
        c = Counter(x.flatten())
        if 0 not in c: return x
        main = c.most_common(1)[0][0]
        res = x.copy()
        res[x == 0] = main
        return res

    def denoise(self, x):
        if x.shape[0] < 3 or x.shape[1] < 3:
            return x
        res = x.copy()
        for i in range(1,x.shape[0]-1):
            for j in range(1,x.shape[1]-1):
                block = x[i-1:i+2, j-1:j+2].flatten()
                c = Counter(block)
                if c[x[i,j]] == 1:
                    res[i,j] = c.most_common(1)[0][0]
        return res

    # ======================
    # SCALE DETECTION
    # ======================
    def detect_scale(self, inp, out):
        ih, iw = inp.shape
        oh, ow = out.shape
        if oh % ih == 0 and ow % iw == 0:
            return (oh // ih, ow // iw)
        return None

    # ======================
    # SCORE
    # ======================
    def score(self, pred, target):
        target = np.array(target)
        if pred.shape != target.shape:
            return 0
        return np.mean(pred == target)

    # ======================
    # SEARCH ENGINE
    # ======================
    def find_best_rule(self, train):
        best_score = -1
        best_rule = None

        for rule in self.rules:
            try:
                scores = []

                for ex in train:
                    inp = np.array(ex['input'])
                    out = np.array(ex['output'])

                    scale = self.detect_scale(inp, out)

                    pred = rule(inp)
                    if scale:
                        pred = np.tile(pred, scale)

                    s = self.score(pred, out)
                    scores.append(s)

                avg = np.mean(scores)

                if avg > best_score:
                    best_score = avg
                    best_rule = rule

            except:
                continue

        # COLOR MAP PRIORITY
        if best_score < 1.0:
            for ex in train:
                m = self.get_color_map(ex['input'], ex['output'])
                if m:
                    def cmap(x, mapping=m):
                        return self.apply_color_map(x, mapping)

                    if all(np.array_equal(cmap(np.array(e['input'])), e['output']) for e in train):
                        return cmap

        return best_rule

    # ======================
    # SOLVER
    # ======================
    def solve(self, task):
        best_rule = self.find_best_rule(task['train'])
        results = []

        for t in task['test']:
            inp = np.array(t['input'])
            attempts = []

            if best_rule:
                try:
                    res = best_rule(inp).tolist()
                    attempts.append(res)
                except:
                    pass

            # fallback
            for fb in [self.identity, self.rotate_90]:
                res = fb(inp).tolist()
                if res not in attempts:
                    attempts.append(res)

            while len(attempts) < 2:
                attempts.append(inp.tolist())

            results.append({
                "attempt_1": attempts[0],
                "attempt_2": attempts[1]
            })

        return results


# ======================
# MAIN
# ======================
input_path = '/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_test_challenges.json'

with open(input_path, 'r') as f:
    tasks = json.load(f)

solver = ARCSolver()

submission = {
    tid: solver.solve(task)
    for tid, task in tasks.items()
}

with open('submission.json', 'w') as f:
    json.dump(submission, f)

print("💎 FINAL ARC SOLVER READY (OBJECT + HEURISTIC HYBRID)")

💎 FINAL ARC SOLVER READY (OBJECT + HEURISTIC HYBRID)
